# Candidate Generation and Pairwise Features

Build the first recommendation step: retrieve 100 candidate users for one target user.

This notebook:

- loads `data/processed_data.csv`
- creates target-candidate pairwise features
- ranks candidates with a simple heuristic baseline
- validates that candidate generation returns exactly 100 users
- keeps the scoring logic explainable before neural ranking


## Why use pairwise features?

Recommendation is about comparing two users, not predicting a single-user label.

Since this dataset has no follow, click, like, or impression labels, the first baseline should be deterministic and explainable.

In [1]:
from pathlib import Path
import json

import pandas as pd

PROCESSED_DATA_CANDIDATES = [
    Path("../data/processed_data.csv"),  # when notebook is run from EDA/
    Path("data/processed_data.csv"),     # when notebook is run from repo root
]

PROCESSED_DATA_PATH = next((path for path in PROCESSED_DATA_CANDIDATES if path.exists()), None)
if PROCESSED_DATA_PATH is None:
    raise FileNotFoundError("Could not find processed_data.csv in data/ or ../data/")

PROCESSED_DATA_PATH


PosixPath('../data/processed_data.csv')

## 1. Load processed data

Use the cleaned dataset created in preprocessing.

In [2]:
users_df = pd.read_csv(PROCESSED_DATA_PATH, dtype={"user_id": "string"})

users_df["interests"] = users_df["interests_deduped"].apply(json.loads)
users_df["interest_set"] = users_df["interests"].apply(set)

print(f"Rows: {len(users_df):,}")
print(f"Columns: {list(users_df.columns)}")
users_df.head()


Rows: 100,000
Columns: ['user_id', 'name', 'gender', 'dob', 'age', 'city', 'country', 'city_country_key', 'interests_raw', 'interests_list', 'interests_deduped', 'interest_count', 'split', 'interests', 'interest_set']


,user_id,name,gender,dob,age,city,country,city_country_key,interests_raw,interests_list,interests_deduped,interest_count,split,interests,interest_set
0,00001,Jesse Lawhorn,Female,1958-10-15,67,Sibolga,Indonesia,"Sibolga, Indonesia","'Movies', 'Fashion', 'Fashion', 'Books'","[""Movies"", ""Fashion"", ""Fashion"", ""Books""]","[""Movies"", ""Fashion"", ""Books""]",3,train,"[Movies, Fashion, Books]","{Books, Fashion, Movies}"
1,00002,Stacy Payne,Female,2004-07-21,22,Al Abyār,Libya,"Al Abyār, Libya","'Gaming', 'Finance and investments', 'Outdoor ...","[""Gaming"", ""Finance and investments"", ""Outdoor...","[""Gaming"", ""Finance and investments"", ""Outdoor...",4,train,"[Gaming, Finance and investments, Outdoor acti...","{Travel, Finance and investments, Gaming, Outd..."
2,00003,Katrina Nicewander,Female,2000-02-07,26,Wādī as Sīr,Jordan,"Wādī as Sīr, Jordan","'DIY and crafts', 'Music', 'Science', 'Fashion'","[""DIY and crafts"", ""Music"", ""Science"", ""Fashion""]","[""DIY and crafts"", ""Music"", ""Science"", ""Fashion""]",4,validation,"[DIY and crafts, Music, Science, Fashion]","{Music, Science, Fashion, DIY and crafts}"
3,00004,Eric Yarbrough,Male,1985-04-14,41,Matera,Italy,"Matera, Italy","'Outdoor activities', 'Cars and automobiles'","[""Outdoor activities"", ""Cars and automobiles""]","[""Outdoor activities"", ""Cars and automobiles""]",2,train,"[Outdoor activities, Cars and automobiles]","{Outdoor activities, Cars and automobiles}"
4,00005,Daniel Adkins,Female,1955-09-18,70,Biruaca,Venezuela,"Biruaca, Venezuela","'Politics', 'History'","[""Politics"", ""History""]","[""Politics"", ""History""]",2,train,"[Politics, History]","{History, Politics}"


## 2. Select a target user

Use one test user as a realistic inference example.

In [3]:
target_user = users_df[users_df["split"] == "test"].iloc[0]
target_user_id = target_user["user_id"]

target_summary = target_user[
    [
        "user_id",
        "name",
        "age",
        "city",
        "country",
        "city_country_key",
        "interests_deduped",
        "split",
    ]
]

target_summary


user_id                                            00029
name                                           Rena Wild
age                                                   43
city                                              Motril
country                                            Spain
city_country_key                           Motril, Spain
interests_deduped    ["History", "Books", "Photography"]
split                                               test
Name: 28, dtype: object

## 3. Build pairwise candidate features

These features compare one target user with every possible candidate user.

In [4]:
def build_candidate_features(target_user: pd.Series, candidate_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build pairwise features between one target user and many candidate users.

    The dataset has no recommendation labels, so these features are not measured
    against a true target yet. They provide an explainable baseline for candidate
    generation and can later become inputs to a neural ranker.
    """
    target_interests = target_user["interest_set"]
    target_interest_count = len(target_interests)

    features = candidate_df.copy()

    features["target_user_id"] = target_user["user_id"]
    features["candidate_user_id"] = features["user_id"]

    features["interest_overlap_count"] = features["interest_set"].apply(
        lambda candidate_interests: len(target_interests.intersection(candidate_interests))
    )
    features["interest_union_count"] = features["interest_set"].apply(
        lambda candidate_interests: len(target_interests.union(candidate_interests))
    )
    features["interest_jaccard_similarity"] = (
        features["interest_overlap_count"] / features["interest_union_count"]
    ).fillna(0.0)

    features["same_city"] = (features["city"] == target_user["city"]).astype(int)
    features["same_country"] = (features["country"] == target_user["country"]).astype(int)
    features["same_city_country_key"] = (
        features["city_country_key"] == target_user["city_country_key"]
    ).astype(int)

    features["age_difference"] = (features["age"] - target_user["age"]).abs()
    features["target_interest_count"] = target_interest_count
    features["candidate_interest_count"] = features["interest_count"]

    return features


## 4. Add baseline retrieval score

This score is a heuristic, not a learned model.

Interest similarity is weighted highest. `same_city_country_key` gets a stronger boost than `same_country` because it is more specific.

### Retrieval weight note

Current baseline weights:

- `0.65` × interest Jaccard similarity
- `0.15` × same country
- `0.20` × same city-country key

This keeps interest similarity as the primary signal while still giving location a useful secondary role. Exact `city_country_key` matches receive a stronger location boost than broad `country` matches because they are more specific.

A full A/B or sensitivity test is intentionally skipped here. The dataset is limited and has no real interaction labels such as follows, clicks, likes, impressions, or accepted recommendations. With this kind of dummy/static dataset, weight tuning can only show how the heuristic ranking changes; it cannot prove recommendation quality. A deeper A/B-style comparison would make more sense with richer behavioral data and more project time.

The retrieval score is not a probability. It is only a ranking score used to create a reasonable top-100 candidate pool before neural ranking.


In [5]:
def add_retrieval_score(features: pd.DataFrame) -> pd.DataFrame:
    """
    Add an explainable baseline score for candidate retrieval.

    The weights are manually chosen because the dataset has no real interaction
    labels. This is used to retrieve a reasonable top-100 candidate pool before
    neural ranking.
    """
    scored = features.copy()

    scored["retrieval_score"] = (
        0.65 * scored["interest_jaccard_similarity"]
        + 0.15 * scored["same_country"]
        + 0.20 * scored["same_city_country_key"]
    )

    return scored


def generate_candidates(target_user_id: str, users: pd.DataFrame, top_k: int = 100) -> pd.DataFrame:
    """
    Return the top-k candidate users for a target user.

    The target user is excluded from the candidate pool. The full remaining user
    table is scored in memory, which matches the assessment requirement for fast
    candidate generation without live API disk I/O.
    """
    matching_targets = users[users["user_id"] == target_user_id]
    if matching_targets.empty:
        raise ValueError(f"Unknown target_user_id: {target_user_id}")

    target = matching_targets.iloc[0]
    candidate_pool = users[users["user_id"] != target_user_id]

    features = build_candidate_features(target, candidate_pool)
    scored = add_retrieval_score(features)

    ranked = scored.sort_values(
        by=[
            "retrieval_score",
            "interest_overlap_count",
            "same_city_country_key",
            "same_country",
        ],
        ascending=[False, False, False, False],
    )

    return ranked.head(top_k).reset_index(drop=True)


## 5. Generate top-100 candidates

Run candidate generation for the sample target user.

In [6]:
top_100_candidates = generate_candidates(target_user_id, users_df, top_k=100)

candidate_display_columns = [
    "target_user_id",
    "candidate_user_id",
    "name",
    "age",
    "city",
    "country",
    "interests_deduped",
    "interest_overlap_count",
    "interest_jaccard_similarity",
    "same_city",
    "same_country",
    "same_city_country_key",
    "age_difference",
    "retrieval_score",
]

top_100_candidates[candidate_display_columns].head(10)


,target_user_id,candidate_user_id,name,age,city,country,interests_deduped,interest_overlap_count,interest_jaccard_similarity,same_city,same_country,same_city_country_key,age_difference,retrieval_score
0,00029,14313,Mark Boudreau,53,Owensboro,United States,"[""Photography"", ""History"", ""Books""]",3,1.000000,0,0,0,10,0.650000
1,00029,28264,Rosa Smith,66,La Unión,El Salvador,"[""Photography"", ""History"", ""Books""]",3,1.000000,0,0,0,23,0.650000
2,00029,49817,Ethel Muniz,66,Bobingen,Germany,"[""History"", ""Books"", ""Photography""]",3,1.000000,0,0,0,23,0.650000
3,00029,74128,Marcia Sweene,22,Chiba,Japan,"[""Photography"", ""History"", ""Books""]",3,1.000000,0,0,0,21,0.650000
4,00029,74160,Marisol Turner,24,Danville,United States,"[""Photography"", ""Books"", ""History""]",3,1.000000,0,0,0,19,0.650000
5,00029,22592,Rafael Hartrick,22,Porta,Spain,"[""History"", ""Books""]",2,0.666667,0,1,0,21,0.583333
6,00029,37465,Cherie Smith,69,la Verneda i la Pau,Spain,"[""History"", ""Photography""]",2,0.666667,0,1,0,26,0.583333
7,00029,48854,Norman Ballard,45,Villanueva del Pardillo,Spain,"[""Books"", ""History""]",2,0.666667,0,1,0,2,0.583333
8,00029,58460,Jared Hines,27,Vila-real,Spain,"[""Photography"", ""History""]",2,0.666667,0,1,0,16,0.583333
9,00029,02911,Peggy Hopkins,38,Lakshmīpur,Bangladesh,"[""Books"", ""History"", ""DIY and crafts"", ""Photog...",3,0.750000,0,0,0,5,0.487500


## 6. Validate candidate generation

Confirm that the retrieval function returns a clean top-100 candidate pool.

In [7]:
candidate_validation_report = pd.DataFrame(
    [
        {"check": "returns exactly 100 candidates", "passed": len(top_100_candidates) == 100, "details": len(top_100_candidates)},
        {"check": "target user is excluded", "passed": target_user_id not in set(top_100_candidates["candidate_user_id"]), "details": target_user_id},
        {"check": "candidate IDs are unique", "passed": top_100_candidates["candidate_user_id"].is_unique, "details": int(top_100_candidates["candidate_user_id"].duplicated().sum())},
        {"check": "all candidates share at least one interest", "passed": (top_100_candidates["interest_overlap_count"] > 0).all(), "details": int((top_100_candidates["interest_overlap_count"] == 0).sum())},
        {"check": "scores are sorted descending", "passed": top_100_candidates["retrieval_score"].is_monotonic_decreasing, "details": [float(top_100_candidates["retrieval_score"].max()), float(top_100_candidates["retrieval_score"].min())]},
    ]
)

candidate_validation_report


,check,passed,details
0,returns exactly 100 candidates,True,100
1,target user is excluded,True,00029
2,candidate IDs are unique,True,0
3,all candidates share at least one interest,True,0
4,scores are sorted descending,True,"[0.65, 0.43333333333333335]"


## 7. Inspect score breakdown

Check whether candidates are being selected mainly through interests, location, or both.

In [8]:
score_breakdown = pd.DataFrame(
    [
        {"metric": "min_retrieval_score", "value": float(top_100_candidates["retrieval_score"].min())},
        {"metric": "max_retrieval_score", "value": float(top_100_candidates["retrieval_score"].max())},
        {"metric": "avg_interest_overlap", "value": float(top_100_candidates["interest_overlap_count"].mean())},
        {"metric": "avg_interest_jaccard", "value": float(top_100_candidates["interest_jaccard_similarity"].mean())},
        {"metric": "same_country_candidates", "value": int(top_100_candidates["same_country"].sum())},
        {"metric": "same_city_country_candidates", "value": int(top_100_candidates["same_city_country_key"].sum())},
    ]
)

score_breakdown


,metric,value
0,min_retrieval_score,0.433333
1,max_retrieval_score,0.650000
2,avg_interest_overlap,2.110000
3,avg_interest_jaccard,0.670000
4,same_country_candidates,15.000000
5,same_city_country_candidates,0.000000


## Executed baseline summary

- Sample target user: `00029`, from `Motril, Spain`.
- Target interests: `History`, `Books`, `Photography`.
- Candidate generation returned exactly `100` candidates.
- The target user was excluded from the candidate pool.
- All returned candidates share at least one interest with the target user.
- Retrieval scores ranged from `0.4333` to `0.65`.
- Average interest overlap in the top 100 was `2.11`.
- `15` candidates were from the same country.
- `0` candidates had the same `city_country_key`.

Current behavior: the baseline remains interest-driven, restores useful country-level signal, and gives exact `city_country_key` matches a stronger boost than broad `country` matches.


## Candidate-generation insights

- Candidate generation should use pairwise target-candidate features, not single-user labels.
- The current retrieval score is a baseline heuristic because the dataset has no interaction labels.
- Interest similarity should drive retrieval because interests are the strongest available behavioral signal.
- `country`, `city`, and `city_country_key` should stay available as separate location signals.
- The next modeling step can use these pairwise features to create synthetic affinity labels or train a neural ranker.
- Heuristic weight tuning should be treated as a limitation, not as evidence of true recommendation accuracy.
